In [11]:
from numpy import*
import math

In [12]:
def loadDataSet():
  return [[1, 3, 4], [2, 3, 5], [1, 2, 3, 5], [2, 5]]

In [13]:
def createCl(dataSet):
  Cl=[]
  for transaction in dataSet:
    for item in transaction:
      if not [item] in Cl:
        Cl.append([item])
  Cl.sort()
  return list(map(frozenset,Cl))

In [14]:
def scanD(D,Ck,minSupport):
  ssCnt={}
  for tid in D:
    for can in Ck:
      if can.issubset(tid):
        ssCnt[can]=ssCnt.get(can,0)+1
  numItems=float(len(D))
  retList,supportData=[],{}
  for key in ssCnt:
    support=ssCnt[key]/numItems
    if support >= minSupport:
      retList.insert(0,key)
    supportData[key]=support
  return retList,supportData

In [15]:
def aprioriGen(Lk,k):
  retList=[]
  lenLk=len(Lk)
  for i in range(lenLk):
    for j in range(i+1,lenLk):
      L1=list(Lk[i])[:k-2]
      L2=list(Lk[j])[:k-2]
      L1.sort()
      L2.sort()
      if L1==L2:
        retList.append(Lk[i]|Lk[j])
  return retList

In [16]:
def apriori(dataSet,minSupport=0.5):
  C1=createCl(dataSet)
  D=list(map(set,dataSet))
  L1,supportData=scanD(D,C1,minSupport)
  L=[L1]
  k=2
  while (len(L[k-2]) > 0):
    Ck=aprioriGen(L[k-2],k)
    Lk,supK=scanD(D,Ck,minSupport)
    supportData.update(supK)
    L.append(Lk)
    k+=1
  return L,supportData

In [17]:
def calcConf(freqSet,H,supportData,brl,minConf=0.7):
  prunedH=[]
  for conseq in H:
    conf=supportData[freqSet]/supportData[freqSet-conseq]
    if conf >= minConf:
      print(freqSet-conseq,'-->',conseq,'conf:',conf)
      brl.append((freqSet-conseq,conseq,conf))
      prunedH.append(conseq)
  return prunedH

In [18]:
def rulesFromConseq(freqSet,H,supportData,brl,minConf=0.7):
  m=len(H[0])
  if (len(freqSet)>(m+1)):
    Hmp1=aprioriGen(H,m+1)
    Hmp1=calcConf(freqSet,Hmp1,supportData,brl,minConf)
    if (len(Hmp1)>1):
      rulesFromConseq(freqSet,Hmp1,supportData,brl,minConf)


In [19]:
def generateRules(L,supportData,minConf=0.7):
  bigRuleList=[]
  for i in range(1,len(L)):
    for freqSet in L[i]:
      H1=[frozenset([item]) for item in freqSet]
      if (i>1):
        rulesFromConseq(freqSet,H1,supportData,bigRuleList,minConf)
      else:
        calcConf(freqSet,H1,supportData,bigRuleList,minConf)
  return bigRuleList

In [20]:
dataSet = loadDataSet()
L, supportData = apriori(dataSet, minSupport=0.5)
rules = generateRules(L, supportData, minConf=0.7)
print('\nGenerated Rules:')
for rule in rules:
    print(f"{set(rule[0])} --> {set(rule[1])} (conf: {rule[2]:.2f})")

frozenset({5}) --> frozenset({2}) conf: 1.0
frozenset({2}) --> frozenset({5}) conf: 1.0
frozenset({1}) --> frozenset({3}) conf: 1.0

Generated Rules:
{5} --> {2} (conf: 1.00)
{2} --> {5} (conf: 1.00)
{1} --> {3} (conf: 1.00)


### Comprehensive Demonstration of the Apriori Algorithm

Below, we run the completed Apriori algorithm with different parameter settings to discover more association rules from our dataset.

In [21]:
def run_apriori_demo(support_threshold, confidence_threshold):
    print(f"=== Apriori with minSupport={support_threshold} and minConf={confidence_threshold} ===")

    # 1. Load Data
    dataset = loadDataSet()

    # 2. Run Apriori to find frequent itemsets
    L, support_data = apriori(dataset, minSupport=support_threshold)

    # 3. Generate association rules
    rules = generateRules(L, support_data, minConf=confidence_threshold)

    # 4. Display Results
    print(f"\nFound {sum(len(level) for level in L)} frequent itemsets.")
    print(f"Generated {len(rules)} association rules:\n")
    for idx, rule in enumerate(rules, 1):
        lhs = set(rule[0])
        rhs = set(rule[1])
        conf = rule[2]
        print(f"Rule #{idx}: {lhs} ---> {rhs} (Confidence: {conf:.2%})")
    print("\n" + "="*50 + "\n")

# Run with lowered thresholds to discover more relationships
run_apriori_demo(support_threshold=0.3, confidence_threshold=0.5)

=== Apriori with minSupport=0.3 and minConf=0.5 ===
frozenset({3}) --> frozenset({2}) conf: 0.6666666666666666
frozenset({2}) --> frozenset({3}) conf: 0.6666666666666666
frozenset({5}) --> frozenset({3}) conf: 0.6666666666666666
frozenset({3}) --> frozenset({5}) conf: 0.6666666666666666
frozenset({5}) --> frozenset({2}) conf: 1.0
frozenset({2}) --> frozenset({5}) conf: 1.0
frozenset({3}) --> frozenset({1}) conf: 0.6666666666666666
frozenset({1}) --> frozenset({3}) conf: 1.0
frozenset({5}) --> frozenset({2, 3}) conf: 0.6666666666666666
frozenset({3}) --> frozenset({2, 5}) conf: 0.6666666666666666
frozenset({2}) --> frozenset({3, 5}) conf: 0.6666666666666666

Found 9 frequent itemsets.
Generated 11 association rules:

Rule #1: {3} ---> {2} (Confidence: 66.67%)
Rule #2: {2} ---> {3} (Confidence: 66.67%)
Rule #3: {5} ---> {3} (Confidence: 66.67%)
Rule #4: {3} ---> {5} (Confidence: 66.67%)
Rule #5: {5} ---> {2} (Confidence: 100.00%)
Rule #6: {2} ---> {5} (Confidence: 100.00%)
Rule #7: {3} -

/tmp/ipykernel_2195/2425725178.py:14: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  print(f"\nFound {sum(len(level) for level in L)} frequent itemsets.")
